# Self-attention, geometrically
Two-word sentences with 2-number embeddings, so that every vector can be drawn. All numbers are hypothetical (chosen by hand, not trained).

In [1]:
import numpy as np
import pandas as pd
np.set_printoptions(precision=3, suppress=True)

E = {"money": np.array([2.0, 7.0]), "bank": np.array([7.0, 3.0]), "river": np.array([8.0, -2.0])}
W_Q = np.array([[0.2, 0.1], [0.1, 0.3]])
W_K = np.array([[0.3, 0.0], [0.1, 0.2]])
W_V = np.array([[0.9, 0.2], [0.1, 0.8]])

def softmax(s):
    e = np.exp(s - s.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

def self_attention(words):
    X = np.stack([E[w] for w in words])          # one row per word
    Q, K, V = X @ W_Q, X @ W_K, X @ W_V
    S = Q @ K.T                                  # similarity scores
    A = softmax(S / np.sqrt(2))                  # d_k = 2
    return X, Q, K, V, S, A, A @ V

## "money bank": every step

In [2]:
rows = []
for words in (["money", "bank"], ["river", "bank"]):
    X, Q, K, V, S, A, Y = self_attention(words)
    print(" ".join(words))
    for i, w in enumerate(words):
        print(f"  {w:6s} e {X[i]}  q {Q[i]}  k {K[i]}  v {V[i]}")
    print("  scores\n", S, "\n  scaled\n", S / np.sqrt(2), "\n  weights\n", A, "\n  outputs y\n", Y)
    for i, w in enumerate(words):
        for kind, M in (("e", X), ("q", Q), ("k", K), ("v", V), ("y", Y)):
            rows.append(dict(sentence=" ".join(words), word=w, kind=kind, x=M[i, 0], y=M[i, 1]))
        for j, w2 in enumerate(words):
            rows.append(dict(sentence=" ".join(words), word=w, kind=f"weight_{w2}", x=A[i, j], y=S[i, j]))
pd.DataFrame(rows).to_csv("data/vectors.csv", index=False, float_format="%.4f")

money bank
  money  e [2. 7.]  q [1.1 2.3]  k [1.3 1.4]  v [2.5 6. ]
  bank   e [7. 3.]  q [1.7 1.6]  k [2.4 0.6]  v [6.6 3.8]
  scores
 [[4.65 4.02]
 [4.45 5.04]] 
  scaled
 [[3.288 2.843]
 [3.147 3.564]] 
  weights
 [[0.61  0.39 ]
 [0.397 0.603]] 
  outputs y
 [[4.101 5.141]
 [4.972 4.674]]
river bank
  river  e [ 8. -2.]  q [1.4 0.2]  k [ 2.2 -0.4]  v [7. 0.]
  bank   e [7. 3.]  q [1.7 1.6]  k [2.4 0.6]  v [6.6 3.8]
  scores
 [[3.   3.48]
 [3.1  5.04]] 
  scaled
 [[2.121 2.461]
 [2.192 3.564]] 
  weights
 [[0.416 0.584]
 [0.202 0.798]] 
  outputs y
 [[6.766 2.219]
 [6.681 3.031]]


## The output lies on the segment between the two value vectors
The weights are positive and sum to 1, so y_bank = v_bank + w_money (v_money - v_bank): a point on the segment from v_bank to v_money, a fraction w_money of the way.

In [3]:
for words in (["money", "bank"], ["river", "bank"]):
    X, Q, K, V, S, A, Y = self_attention(words)
    w_other = A[1, 0]
    assert np.allclose(A.sum(1), 1)
    assert np.allclose(Y[1], V[1] + w_other * (V[0] - V[1]))
    moved = np.linalg.norm(Y[1] - V[1])
    print(f"{words[0]:6s} bank: y_bank = {Y[1]}, {w_other:.3f} of the way from v_bank to v_{words[0]}, "
          f"moved {moved:.2f} units")
print("y_bank in the two sentences differs by", np.linalg.norm(self_attention(["money", "bank"])[6][1]
                                                - self_attention(["river", "bank"])[6][1]).round(2))

money  bank: y_bank = [4.972 4.674], 0.397 of the way from v_bank to v_money, moved 1.85 units
river  bank: y_bank = [6.681 3.031], 0.202 of the way from v_bank to v_river, moved 0.77 units
y_bank in the two sentences differs by 2.37
